# 2021 年韩国关税署 10 月 1–20 日原稿来源

冻结官网 HTML、HWP、PDF 三次只读收取，先存原始字节与 NAS 回读，后检查表格。无 SOXX 结果、拟合或账户。

In [1]:
import hashlib,json,subprocess,tempfile
from datetime import datetime,timezone,date
from pathlib import Path
from src.data.nas_market_data import publish_nas_snapshot,materialize_nas_snapshot
root=Path.cwd();card=root/'research/kcs_early20_2021_oct_original_source_20261023';plan=json.loads((card/'plan.json').read_text());sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and not (card/'run_state.json').exists()
raw=root/'data/snapshots/kcs_early20_2021_oct_original_source_20261023';raw.mkdir(parents=True,exist_ok=True);assert not any(raw.iterdir())
state={'status':'started','plan_sha256':sha(card/'plan.json'),'GET_attempts':0,'SOXX_outcome_reads':0,'model_fits':0,'native_accounts':0};(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
receipts=[];failure=None
for spec in plan['exact_requests']:
    state['GET_attempts']+=1;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
    body=raw/spec['name'];headers=raw/(spec['name']+'.headers');started=datetime.now(timezone.utc).isoformat();t=plan['transport']
    cmd=['curl','--http1.1','--noproxy','*','--connect-timeout',str(t['connect_timeout_seconds']),'--max-time',str(t['total_timeout_seconds_each']),'--max-filesize',str(t['max_response_bytes_each']),'--fail-with-body','-sS','-L','-D',str(headers),'-o',str(body),'-A',t['user_agent'],'-w','%{http_code} %{url_effective}',spec['url']]
    try:
        run=subprocess.run(cmd,capture_output=True,text=True,timeout=40);parts=run.stdout.split(' ',1);receipt={'name':spec['name'],'url':spec['url'],'requested_at_utc':started,'received_at_utc':datetime.now(timezone.utc).isoformat(),'curl_exit_code':run.returncode,'http_status':int(parts[0]) if parts[0].isdigit() else None,'final_url':parts[1] if len(parts)>1 else None,'transport_error':run.stderr[:250]}
    except Exception as exc:receipt={'name':spec['name'],'url':spec['url'],'requested_at_utc':started,'received_at_utc':datetime.now(timezone.utc).isoformat(),'transport_exception':type(exc).__name__+':'+str(exc)[:180]}
    receipt.update(body_sha256=sha(body) if body.exists() else None,body_bytes=body.stat().st_size if body.exists() else 0,headers_sha256=sha(headers) if headers.exists() else None,headers_bytes=headers.stat().st_size if headers.exists() else 0)
    receipts.append(receipt);(raw/'capture_receipts.json').write_text(json.dumps(receipts,indent=2)+'\n')
    if receipt.get('curl_exit_code')!=0 or receipt.get('http_status')!=200 or not body.exists() or body.stat().st_size>t['max_response_bytes_each']:
        failure='transport_or_size:'+spec['name'];break
    data=body.read_bytes()
    if spec['name']=='official_index.html':
        if not all(x in data for x in (b'156476286',b'196144721',b'196144722',b'2021.10.21')):failure='page_identity_or_attachment_ids';break
    elif spec['name'].endswith('.hwp'):
        if not data.startswith(bytes.fromhex('D0CF11E0A1B11AE1')):failure='HWP_signature';break
    elif spec['name'].endswith('.pdf'):
        if not data.startswith(b'%PDF-'):failure='PDF_signature';break
manifest={'schema_version':1,'experiment':plan['experiment'],'plan_sha256':state['plan_sha256'],'capture_utc':datetime.now(timezone.utc).isoformat(),'receipts':receipts,'failure':failure,'historical_first_public_bytes_qualified':False}
(raw/'source_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n');(card/'source_receipt.json').write_text(json.dumps(manifest,indent=2)+'\n')
files={str(p.relative_to(root)):'raw' for p in raw.iterdir() if p.is_file()}
try:
    binding=publish_nas_snapshot(workspace_root=root,source_manifest=raw/'source_manifest.json',files=files,as_of=date.today().isoformat(),source_metadata={'experiment':plan['experiment'],'source':'Korea government 2021 Oct1-20 KCS press release'})
    (card/'nas_binding.json').write_text(json.dumps(binding,indent=2)+'\n')
    with tempfile.TemporaryDirectory(prefix='kcs-2021-cold-') as temp:
        readback=materialize_nas_snapshot(binding,workspace_root=Path(temp))
        for obj in binding['objects']:assert sha(Path(temp)/obj['local_path'])==obj['sha256']
    assert binding['verified'] and readback['restored_objects']==len(files)
except Exception as exc:
    failure=(failure+';' if failure else '')+'NAS:'+type(exc).__name__+':'+str(exc)[:180];binding=None
state.update(status='completed' if not failure else 'stopped',failure=failure);(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
result={'decision':'official_2021_KCS_early20_original_page_and_attachments_received_review_pending' if not failure else 'needs_2021_KCS_source_or_NAS','failure':failure,'GET_attempts':state['GET_attempts'],'receipt_count':len(receipts),'NAS_source_id':binding['source_id'] if binding else None,'NAS_readback_verified':bool(binding),'historical_first_public_bytes_qualified':False,'SOXX_outcome_reads':0,'model_fits':0,'native_accounts':0,'strategy_forward_days':0}
(card/'capture_result.json').write_text(json.dumps(result,indent=2)+'\n');print(json.dumps(result))


{"decision": "official_2021_KCS_early20_original_page_and_attachments_received_review_pending", "failure": null, "GET_attempts": 3, "receipt_count": 3, "NAS_source_id": "8ec18dacd29420dc009c8a11666db742313e83bd250b4493b9557fb0e1e26a3e", "NAS_readback_verified": true, "historical_first_public_bytes_qualified": false, "SOXX_outcome_reads": 0, "model_fits": 0, "native_accounts": 0, "strategy_forward_days": 0}


# 2021 年韩国关税署 10 月 1–20 日半导体出口原稿

2021-10-21 韩国政府政策门户的关税署原稿页、所列 HWP 与 PDF 附件均按冻结 URL 一次取得，三份原始响应及收取时钟、哈希已存入 Yanxuan NAS 并远端核对、定向冷恢复。PDF 共两页；第 2 页“主要出口品”表经渲染目视核对，**半导体出口 6,394 百万美元、同比 +23.9%**。页内标明这是 10 月 1–20 日暂定统计，受工作日变化影响，年度统计确定前可能修订。

官网页面日期为 **2021-10-21**。当前 PDF 内部 CreationDate/ModDate 均为 **2021-10-21 09:01:28 KST**，与同日原稿身份一致，但元数据不是独立的首次公开收据，不能证明今天的 PDF/HWP 字节当时就已在线。HWP 的 OLE 文件身份通过，内部元数据和数值单元未解析；PDF 生成时间、HWP 名称或官网日期均不能单独补足这一缺口。详细边界见 `source_review.json`。

这为韩国 1–20 日出口来源补上一份**2021 年官方历史实例**，尚不是完整 2018–2022 历史训练面，更没有定义需求惊喜、建立 Qlib 因子、读取 SOXX 结果、拟合或运行 Stage1 0bp 账户。Goal 的 SOXX 硬门仍未完成。官方来源：[韩国政府原稿页](https://www.korea.kr/briefing/pressReleaseView.do?newsId=156476286)。
